# PG-MAP — One-click Colab quickstart

**NeurIPS 2026** · [github.com/sophialanlan/PG-MAP](https://github.com/sophialanlan/PG-MAP)

This notebook runs **PG-MAP** (Preference-Guided Adaptive MAP) on a small subset of PartiPrompts using a free Colab GPU. It demonstrates:

1. Loading the official PG-MAP custom pipeline from the HuggingFace Hub.
2. Generating with and without PG-MAP at the same seed (side-by-side comparison).
3. Hyperparameter ablation: $\lambda$ (reward weight), $K$ (inner steps), $\eta_z$ (latent step size).

Estimated runtime on a T4: **~5 minutes** for SD 1.5; **~15 minutes** for SDXL. SD3.5-medium requires accepting the Stability AI Community License on huggingface.co before first run.

> **Tip:** in Colab, click `Runtime → Change runtime type → GPU` (T4 free tier is fine for SD 1.5; SDXL benefits from A100 if available).


## 1. Install

Installs PG-MAP at the pinned v1.2.0 tag plus its diffusers/transformers/HF dependencies. ~90 s.


In [ ]:
%%capture
!pip install -q --upgrade pip
!pip install -q git+https://github.com/sophialanlan/PG-MAP@v1.2.0
!pip install -q gradio  # optional; only for the interactive demo at the bottom

In [ ]:
import torch, pgmap
print('PG-MAP version:', pgmap.__version__)
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

## 2. Load the SD 1.5 PG-MAP pipeline

Loaded from the official HuggingFace Hub custom pipeline `sophialan/pg-map-sd15` — this fetches a single thin shim file from HF and dispatches to the full implementation that lives inside the installed `pgmap` package. ~30 s on a T4.


In [ ]:
from diffusers import DiffusionPipeline

pipe = DiffusionPipeline.from_pretrained(
    'runwayml/stable-diffusion-v1-5',
    custom_pipeline='sophialan/pg-map-sd15',
    torch_dtype=torch.float16,
    safety_checker=None,
    requires_safety_checker=False,
).to('cuda')

print('Loaded:', type(pipe).__name__)

## 3. Side-by-side: vanilla SD 1.5 vs PG-MAP

Generates the same prompt twice at the same seed. The first call is vanilla SD 1.5 (PG-MAP off). The second flips on the paper-default PG-MAP config.


In [ ]:
from pgmap import sd15_defaults, FrozenRewardModel
from dataclasses import replace
import matplotlib.pyplot as plt

prompt = 'a phoenix rising from ashes, vivid orange and red feathers, dramatic lighting'
seed = 42

# --- (a) Vanilla SD 1.5 (no PG-MAP) ---
img_baseline = pipe(
    prompt,
    num_inference_steps=30,
    guidance_scale=7.5,
    generator=torch.Generator(device='cuda').manual_seed(seed),
).images[0]

# --- (b) PG-MAP default (joint c+z + PickScore reward, paper Table 1 row 7) ---
cfg = sd15_defaults()
cfg = replace(cfg, seed=seed)
reward = FrozenRewardModel('pickscore', device='cuda')
img_pgmap = pipe(
    prompt,
    pg_map_config=cfg,
    reward_model=reward,
).images[0]

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
axes[0].imshow(img_baseline); axes[0].set_title('Vanilla SD 1.5 (baseline)'); axes[0].axis('off')
axes[1].imshow(img_pgmap);    axes[1].set_title('PG-MAP (paper default)');   axes[1].axis('off')
plt.tight_layout(); plt.show()

## 4. Hyperparameter ablation — $\lambda$ (reward weight)

Sweeps $\lambda \in \{0, 0.05, 0.10, 0.25\}$ at fixed seed. $\lambda=0$ recovers the **reward-free MAP-cz** variant. The productive range in the paper is $[0.05, 0.10]$. Above $\sim 0.25$ the image drifts off-manifold (CLIPScore drops, visible artifacts).


In [ ]:
lambdas = [0.0, 0.05, 0.10, 0.25]
imgs = []
for lam in lambdas:
    cfg = sd15_defaults()
    cfg = replace(cfg, seed=seed)
    cfg.reward.lambda_reward = lam
    cfg.use_reward = lam > 0
    img = pipe(prompt, pg_map_config=cfg, reward_model=reward if lam > 0 else None).images[0]
    imgs.append(img)

fig, axes = plt.subplots(1, 4, figsize=(20, 6))
for ax, lam, img in zip(axes, lambdas, imgs):
    ax.imshow(img); ax.set_title(f'λ = {lam}'); ax.axis('off')
plt.tight_layout(); plt.show()

## 5. (Optional) SDXL — bigger and more compute-hungry

SDXL @ 1024² is the headline backbone for the human-eval study in the paper. ~30 s per image on a T4, ~5 s on an A100. Skip this section if you're on a free T4 and short on time.


In [ ]:
# Free the SD 1.5 VRAM first
del pipe; import gc; gc.collect(); torch.cuda.empty_cache()

from pgmap import sdxl_defaults

pipe_xl = DiffusionPipeline.from_pretrained(
    'stabilityai/stable-diffusion-xl-base-1.0',
    custom_pipeline='sophialan/pg-map-sdxl',
    torch_dtype=torch.float16,
    variant='fp16',
).to('cuda')

cfg_xl = sdxl_defaults()
cfg_xl = replace(cfg_xl, seed=seed)
img_xl = pipe_xl(prompt, pg_map_config=cfg_xl, reward_model=reward).images[0]

plt.figure(figsize=(8, 8)); plt.imshow(img_xl); plt.title('PG-MAP on SDXL'); plt.axis('off'); plt.show()

## 6. (Optional) SD3.5-medium UG-FM — the 91.9 % PickScore row

Needs Stability AI Community License acceptance on [huggingface.co/stabilityai/stable-diffusion-3.5-medium](https://huggingface.co/stabilityai/stable-diffusion-3.5-medium). Once accepted, sign in to the Hub in Colab via `from huggingface_hub import login; login()` and run the cell below.


In [ ]:
# from huggingface_hub import login; login()       # paste a read-access token

# del pipe_xl; gc.collect(); torch.cuda.empty_cache()

# pipe_sd3 = DiffusionPipeline.from_pretrained(
#     'stabilityai/stable-diffusion-3.5-medium',
#     custom_pipeline='sophialan/pg-map-sd3',
#     torch_dtype=torch.float16,
# ).to('cuda')

# img_fm = pipe_sd3(prompt, reward_model=reward).images[0]   # UG-FM defaults
# plt.figure(figsize=(8, 8)); plt.imshow(img_fm); plt.title('UG-FM (SD3.5)'); plt.axis('off'); plt.show()

## Next steps

- Full paper-table reproduction: see [`scripts/reproduce_table*.sh`](https://github.com/sophialanlan/PG-MAP/tree/main/scripts) in the repo (needs ~3–20 GPU-hours per table on H200).
- Interactive demo: [huggingface.co/spaces/sophialan/pg-map-demo](https://huggingface.co/spaces/sophialan/pg-map-demo).
- Per-row YAML configs (each paper-table cell): [`configs/`](https://github.com/sophialanlan/PG-MAP/tree/main/configs).

## Citation

```bibtex
@inproceedings{sun2026pgmap,
  title={{PG-MAP}: Joint {MAP} Optimization for Inference-Time Alignment of Diffusion and Flow-Matching Models},
  author={Sun, Ruolan and Polak, Pawel},
  booktitle={NeurIPS},
  year={2026}
}
```